# Extra 1 - Sintaxe de `nn.Embedding`

Antes de treinar qualquer coisa, o objetivo aqui e ficar fluente na camada
`torch.nn.Embedding`: como criar, que formato entra e sai, o que muda com
`padding_idx`, e como carregar uma matriz ja pronta (pre-treinada) e
congelar/descongelar os pesos.

Nada e baixado: sao vetores pequenos criados na hora, so para ver os formatos.

Tente resolver antes de olhar `exercicios_1_nn_embedding_sintaxe_solucoes.ipynb`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0);

## 1.1 Criar a camada

Crie `emb = nn.Embedding(num_embeddings=10, embedding_dim=4)` (um vocabulario
de 10 tokens, cada um virando um vetor de tamanho 4).

Imprima `emb.weight.shape` e `emb.weight.dtype`. Essa matriz `(10, 4)` e a
tabela de consulta: a linha `i` e o vetor do token `i`.

In [ ]:
# 1.1

## 1.2 Lookup: o que entra e o que sai

A camada recebe **indices inteiros** (`torch.long`), nunca one-hot.

1. Passe `torch.tensor([1])` e imprima o `.shape` da saida.
2. Passe um batch `torch.tensor([[1, 2, 3], [4, 5, 0]])` (2 frases de 3 tokens)
   e imprima o `.shape` da saida.

Confirme a regra: entrada `(batch, seq_len)` -> saida `(batch, seq_len, embedding_dim)`.

In [ ]:
# 1.2

## 1.3 `padding_idx`

Ao montar batches, frases curtas sao completadas com um token de padding
(normalmente o indice 0). `padding_idx=0` faz a linha 0 ser um vetor de zeros
e **nao receber gradiente** no treino.

1. Crie `emb_pad = nn.Embedding(10, 4, padding_idx=0)`.
2. Imprima `emb_pad.weight[0]` (deve ser tudo zero).
3. Imprima `emb_pad.weight[1]` (uma linha normal, aleatoria).

In [ ]:
# 1.3

## 1.4 Carregar uma matriz pronta com `from_pretrained`

No projeto final voce vai ter uma matriz de embeddings ja treinada (NumPy) e
vai querer coloca-la dentro de um modelo.

1. Crie `mat = np.random.randn(6, 3).astype(np.float32)`.
2. `emb_pt = nn.Embedding.from_pretrained(torch.from_numpy(mat))`.
3. Confirme que `emb_pt.weight` e igual a `mat` (use `torch.allclose`).
4. Imprima `emb_pt.weight.requires_grad` — por padrao `from_pretrained`
   **congela** os pesos (`freeze=True`).

In [ ]:
# 1.4

## 1.5 Congelar / descongelar

1. Crie de novo com `nn.Embedding.from_pretrained(torch.from_numpy(mat), freeze=False)`
   e imprima `requires_grad` (agora `True` — os embeddings vao ser ajustados
   junto com o resto do modelo, "fine-tuning").
2. Pegue essa camada e **congele na mao**: `emb_ft.weight.requires_grad = False`.
   Imprima de novo para confirmar.

In [ ]:
# 1.5

## 1.6 Media dos embeddings de uma frase (com mascara de padding)

Um jeito simples de transformar uma frase (varios vetores) em **um** vetor e
tirar a media dos embeddings dos tokens — ignorando o padding.

Dado `ids = torch.tensor([[2, 3, 4, 0, 0]])` e `emb_pad` (do 1.3):

1. `vecs = emb_pad(ids)` -> formato `(1, 5, 4)`.
2. `mask = (ids != 0)` -> `(1, 5)` booleano. Deixe `(1, 5, 1)` com `.unsqueeze(-1)`.
3. Soma dos vetores validos: `(vecs * mask).sum(dim=1)`.
4. Divida pelo numero de tokens validos: `mask.sum(dim=1)`.
5. Confira que o resultado tem formato `(1, 4)` e que bate com a media
   manual de `emb_pad.weight[[2, 3, 4]]`.

In [ ]:
# 1.6

## Resumo

- `nn.Embedding(V, D)` = tabela `(V, D)`; recebe indices `long`, devolve
  `(..., D)` no lugar de cada indice.
- `padding_idx=0`: linha de zeros que nao treina.
- `nn.Embedding.from_pretrained(tensor)`: carrega matriz pronta; `freeze=True`
  por padrao, `freeze=False` para fine-tuning.
- Media com mascara = frase -> vetor unico (usado no Extra 3).